# Итоговый CatBoost и CRM, участник 1

В этом ноутбуке собираем финальный запуск части участника 1 на всем доступном датасете.

Новый перебор гиперпараметров не выполняется. Используем выводы предыдущих ноутбуков:

- train_capped_target для точного прогноза в секундах
- train_quantile_04 как компромиссный осторожный режим
- train_engagement_quantile_035 для short-risk и retention сценариев

Дополнительно обучаем отдельный классификатор churn 7d и интервальные модели для прикладного CRM выхода.

## Режим запуска

По умолчанию MAX_ROWS равен None, поэтому используется весь датасет.

Полный запуск может занять заметное время. Для быстрой технической проверки можно временно поставить небольшое число строк, например 5000. После проверки нужно вернуть None и выполнить ноутбук заново.

In [1]:
MAX_ROWS = None
SAVE_MODELS = True
MODEL_VERSION = 'crm_full_v1'

## Импорты

Ноутбук должен запускаться независимо сверху вниз из папки участника или из корня репозитория. Здесь находим корень проекта и подключаем preprocessing и общий CRM модуль.

In [2]:
from pathlib import Path
import json
import sys
import time

import numpy as np
import pandas as pd

repo_root = Path.cwd()
if repo_root.name != 'ml_in_gamedev_project':
    for p in [repo_root, *repo_root.parents]:
        if p.name == 'ml_in_gamedev_project':
            repo_root = p
            break

if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

from catboost import CatBoostClassifier, CatBoostRegressor
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score,
)

from preprocessing.preprocessing import (
    CRM_TARGET,
    DEFAULT_TARGET,
    build_feature_list,
    load_data,
    prepare_for_targets,
    regression_metrics,
    TargetTransform,
)
from crm.crm_logic import build_crm_payload, crm_risk_score

pd.set_option('display.max_columns', 300)

## Загрузка данных

Загружаем исходный parquet полностью. На этом шаге данные еще не ограничиваются по числу строк.

In [3]:
df = load_data()
df.shape

(3438527, 84)

## CRM таргеты на горизонте 7 дней

Для каждой сессии игрока смотрим только вперед на 7 дней и исключаем текущую сессию.

Строим:

- future_sessions_mean_playtime_7d — средняя длительность будущих сессий
- future_sessions_count_7d — число будущих сессий
- churn_7d — признак отсутствия будущих сессий

Строки в конце временного диапазона получают пропуски, потому что для них полное недельное окно еще не наблюдается.

In [4]:
def add_crm_retention_targets(
    df_in,
    installation_col='installation_id',
    time_col='start',
    duration_col='duration_seconds',
    horizon_days=7,
):
    x = df_in.copy()
    x[time_col] = pd.to_datetime(x[time_col], errors='coerce')
    x = x[x[time_col].notna() & x[installation_col].notna()].copy()
    x = x.sort_values([installation_col, time_col]).reset_index(drop=True)

    delta_ns = np.int64(horizon_days * 24 * 3600 * 1_000_000_000)
    dur = pd.to_numeric(x[duration_col], errors='coerce').fillna(0.0).to_numpy(dtype=np.float64)
    fut_play = np.zeros(len(x), dtype=np.float64)
    fut_cnt = np.zeros(len(x), dtype=np.int32)

    for _, idx in x.groupby(installation_col, sort=False).indices.items():
        idx = np.asarray(idx, dtype=np.int64)
        t = x.iloc[idx][time_col].values.astype('datetime64[ns]').astype(np.int64)
        d = dur[idx]
        n = len(idx)

        nxt = np.arange(n, dtype=np.int64) + 1
        right = np.searchsorted(t, t + delta_ns, side='right')
        fut_cnt[idx] = np.maximum(right - nxt, 0)

        csum = np.concatenate(([0.0], np.cumsum(d)))
        fut_play[idx] = csum[right] - csum[np.minimum(nxt, n)]

    max_t = x[time_col].max().value
    observed = x[time_col].values.astype('datetime64[ns]').astype(np.int64) <= max_t - delta_ns
    mean7 = np.divide(fut_play, np.maximum(fut_cnt, 1))

    x['future_sessions_count_7d'] = np.where(observed, fut_cnt, np.nan)
    x[CRM_TARGET] = np.where(observed, mean7, np.nan)
    x['churn_7d'] = np.where(observed, (fut_cnt == 0).astype(int), np.nan)
    return x

In [5]:
crm_df = add_crm_retention_targets(df)

target_check = pd.DataFrame({
    'target': [DEFAULT_TARGET, CRM_TARGET, 'churn_7d'],
    'rows': [crm_df[c].notna().sum() for c in [DEFAULT_TARGET, CRM_TARGET, 'churn_7d']],
    'na_rate': [crm_df[c].isna().mean() for c in [DEFAULT_TARGET, CRM_TARGET, 'churn_7d']],
    'mean': [crm_df[c].mean(skipna=True) for c in [DEFAULT_TARGET, CRM_TARGET, 'churn_7d']],
    'median': [crm_df[c].median(skipna=True) for c in [DEFAULT_TARGET, CRM_TARGET, 'churn_7d']],
})
target_check

,target,rows,na_rate,mean,median
0,target_next_session_length_sec,3438527,0.000000,680.616614,300.0
1,future_sessions_mean_playtime_7d,3367930,0.020531,587.948784,462.0
2,churn_7d,3367930,0.020531,0.087783,0.0


## Подготовка двух регрессионных задач

Для каждого таргета формируем отдельный временной split 70 / 15 / 15. При MAX_ROWS равном None ограничения по числу строк нет.

Будущие значения, таргеты, идентификаторы и временные поля не попадают в признаки.

In [6]:
limit = 0 if MAX_ROWS is None else int(MAX_ROWS)
packs = prepare_for_targets(
    crm_df,
    target_cols=[DEFAULT_TARGET, CRM_TARGET],
    max_rows=limit,
)

pd.DataFrame([
    {
        'target': t,
        'train_rows': len(p.x_train),
        'val_rows': len(p.x_val),
        'test_rows': len(p.x_test),
        'features': len(p.feature_cols),
        'cat_features': len(p.cat_cols),
    }
    for t, p in packs.items()
])

,target,train_rows,val_rows,test_rows,features,cat_features
0,target_next_session_length_sec,2406968,515779,515780,74,14
1,future_sessions_mean_playtime_7d,2357551,505189,505190,74,14


## Зафиксированные гиперпараметры

Параметры выбраны по итогам предыдущего CatBoost тюнинга. Здесь мы не ищем новые значения, а честно проверяем финальные режимы на полном объеме данных.

Для всех трех регрессионных стратегий используем единый around-best профиль: глубина 7, learning rate 0.05, 1100 итераций и ранняя остановка по validation.

In [7]:
FINAL_REG_PARAMS = {
    'depth': 7,
    'learning_rate': 0.05,
    'l2_leaf_reg': 3,
    'iterations': 1100,
    'bagging_temperature': 1.0,
    'subsample': 0.70,
    'random_strength': 2.0,
    'border_count': 64,
    'od_type': 'Iter',
    'od_wait': 80,
    'random_seed': 42,
    'verbose': False,
    'thread_count': -1,
}

REG_MODES = [
    {'mode': 'train_capped_target', 'kind': 'capped'},
    {'mode': 'train_quantile_04', 'kind': 'quantile', 'alpha': 0.40},
    {'mode': 'train_engagement_quantile_035', 'kind': 'quantile', 'alpha': 0.35},
]

## Дополнительные регрессионные метрики

К набору из preprocessing добавляем WMAPE и TailWeightedMAE. Первая метрика показывает ошибку относительно масштаба таргета, вторая увеличивает вес ошибок на длинном хвосте.

In [8]:
def wmape(y_true, y_pred):
    y = np.asarray(y_true, dtype=float)
    p = np.asarray(y_pred, dtype=float)
    return float(np.abs(y - p).sum() / max(np.abs(y).sum(), 1e-9))


def tail_weighted_mae(y_true, y_pred, q=0.90, tail_weight=2.0):
    y = np.asarray(y_true, dtype=float)
    p = np.asarray(y_pred, dtype=float)
    cap = np.quantile(y, q)
    w = np.where(y >= cap, tail_weight, 1.0)
    return float(np.average(np.abs(y - p), weights=w))


def metric_pack(y_true, y_pred):
    out = regression_metrics(y_true, y_pred)
    out['wmape'] = wmape(y_true, y_pred)
    out['tail_weighted_mae_q90_w2'] = tail_weighted_mae(y_true, y_pred)
    return out

## Обучение трех итоговых регрессионных стратегий

Для каждого таргета обучаются три модели. train_capped_target ограничивает хвост p995. Два квантильных режима используют разные уровни осторожности.

Сохраняем реальные метрики train, validation и test, чтобы видеть качество и возможное переобучение.

In [9]:
def fit_regression_mode(p, cfg):
    params = FINAL_REG_PARAMS.copy()
    if cfg['kind'] == 'quantile':
        params['loss_function'] = f"Quantile:alpha={cfg['alpha']}"
    else:
        params['loss_function'] = 'MAE'
    params['eval_metric'] = 'MAE'

    transform = TargetTransform('p995' if cfg['kind'] == 'capped' else 'raw')
    transform.fit(p.y_train)
    ytr = transform.transform(p.y_train)

    model = CatBoostRegressor(**params)
    t0 = time.time()
    model.fit(
        p.x_train,
        ytr,
        cat_features=p.cat_cols,
        eval_set=(p.x_val, transform.transform(p.y_val)),
        use_best_model=True,
    )
    fit_sec = time.time() - t0

    pred_tr = transform.inverse(model.predict(p.x_train))
    pred_va = transform.inverse(model.predict(p.x_val))
    pred_te = transform.inverse(model.predict(p.x_test))
    return model, pred_tr, pred_va, pred_te, fit_sec

In [12]:
reg_rows = []
reg_models = {}

In [13]:
mode_name = "train_capped_target"
cfg = next(cfg for cfg in REG_MODES if cfg["mode"] == mode_name)

for target_col, p in packs.items():
    model, ptr, pva, pte, fit_sec = fit_regression_mode(p, cfg)
    reg_models[(target_col, cfg["mode"])] = model

    row = {
        "target": target_col,
        "mode": cfg["mode"],
        "fit_sec": fit_sec,
    }

    for prefix, y, pred in [
        ("train_", p.y_train, ptr),
        ("val_", p.y_val, pva),
        ("test_", p.y_test, pte),
    ]:
        for k, v in metric_pack(y, pred).items():
            row[prefix + k] = v

    reg_rows.append(row)

reg_res = (
    pd.DataFrame(reg_rows)
    .drop_duplicates(subset=["target", "mode"], keep="last")
    .sort_values(["target", "test_mae"])
    .reset_index(drop=True)
)

reg_res

,target,mode,fit_sec,train_mae,train_medae,train_p70_abs_error,train_p90_abs_error,train_r2,train_product_mae,train_engagement_risk_mae,train_small_mae,train_normal_mae,train_long_mae,train_wmape,train_tail_weighted_mae_q90_w2,val_mae,val_medae,val_p70_abs_error,val_p90_abs_error,val_r2,val_product_mae,val_engagement_risk_mae,val_small_mae,val_normal_mae,val_long_mae,val_wmape,val_tail_weighted_mae_q90_w2,test_mae,test_medae,test_p70_abs_error,test_p90_abs_error,test_r2,test_product_mae,test_engagement_risk_mae,test_small_mae,test_normal_mae,test_long_mae,test_wmape,test_tail_weighted_mae_q90_w2
0,future_sessions_mean_playtime_7d,train_capped_target,807.748157,248.659596,144.811009,264.541877,563.002237,0.355720,169.939350,171.951615,142.974194,185.454975,935.795151,0.422667,312.081477,250.502984,141.573665,261.335829,575.978064,0.358644,167.319364,169.490673,130.728380,193.018711,937.977335,0.428136,315.144132,251.024264,142.663518,261.291366,575.839577,0.367384,166.815525,169.964985,133.520267,190.776847,941.949903,0.426110,316.178570
1,target_next_session_length_sec,train_capped_target,1855.794127,532.522333,247.207745,453.982670,1213.471391,0.026725,252.973451,262.715633,212.946364,310.771503,1965.238078,0.782840,723.136644,538.646053,235.876289,444.077759,1205.657169,0.022543,246.627598,256.597849,201.900684,316.468433,2064.185521,0.786807,740.522573,531.317647,242.942723,450.142621,1206.747477,0.027709,250.872921,260.609509,208.225719,315.559087,1983.703216,0.783222,723.162917


In [14]:
mode_name = "train_quantile_04"
cfg = next(cfg for cfg in REG_MODES if cfg["mode"] == mode_name)

for target_col, p in packs.items():
    model, ptr, pva, pte, fit_sec = fit_regression_mode(p, cfg)
    reg_models[(target_col, cfg["mode"])] = model

    row = {
        "target": target_col,
        "mode": cfg["mode"],
        "fit_sec": fit_sec,
    }

    for prefix, y, pred in [
        ("train_", p.y_train, ptr),
        ("val_", p.y_val, pva),
        ("test_", p.y_test, pte),
    ]:
        for k, v in metric_pack(y, pred).items():
            row[prefix + k] = v

    reg_rows.append(row)

reg_res = (
    pd.DataFrame(reg_rows)
    .drop_duplicates(subset=["target", "mode"], keep="last")
    .sort_values(["target", "test_mae"])
    .reset_index(drop=True)
)

reg_res

,target,mode,fit_sec,train_mae,train_medae,train_p70_abs_error,train_p90_abs_error,train_r2,train_product_mae,train_engagement_risk_mae,train_small_mae,train_normal_mae,train_long_mae,train_wmape,train_tail_weighted_mae_q90_w2,val_mae,val_medae,val_p70_abs_error,val_p90_abs_error,val_r2,val_product_mae,val_engagement_risk_mae,val_small_mae,val_normal_mae,val_long_mae,val_wmape,val_tail_weighted_mae_q90_w2,test_mae,test_medae,test_p70_abs_error,test_p90_abs_error,test_r2,test_product_mae,test_engagement_risk_mae,test_small_mae,test_normal_mae,test_long_mae,test_wmape,test_tail_weighted_mae_q90_w2
0,future_sessions_mean_playtime_7d,train_capped_target,807.748157,248.659596,144.811009,264.541877,563.002237,0.355720,169.939350,171.951615,142.974194,185.454975,935.795151,0.422667,312.081477,250.502984,141.573665,261.335829,575.978064,0.358644,167.319364,169.490673,130.728380,193.018711,937.977335,0.428136,315.144132,251.024264,142.663518,261.291366,575.839577,0.367384,166.815525,169.964985,133.520267,190.776847,941.949903,0.426110,316.178570
1,future_sessions_mean_playtime_7d,train_quantile_04,755.880693,257.219048,140.175254,263.728715,602.212944,0.285027,157.819646,165.452109,111.268523,197.864910,1047.087701,0.437216,330.034060,259.819766,137.454177,262.899276,617.177723,0.286597,156.306946,163.386136,100.811035,206.294378,1050.437437,0.444059,333.978900,260.680396,138.524799,264.295701,617.949029,0.295100,156.060127,164.059470,102.564301,204.892126,1054.275137,0.442501,335.326140
2,target_next_session_length_sec,train_capped_target,1855.794127,532.522333,247.207745,453.982670,1213.471391,0.026725,252.973451,262.715633,212.946364,310.771503,1965.238078,0.782840,723.136644,538.646053,235.876289,444.077759,1205.657169,0.022543,246.627598,256.597849,201.900684,316.468433,2064.185521,0.786807,740.522573,531.317647,242.942723,450.142621,1206.747477,0.027709,250.872921,260.609509,208.225719,315.559087,1983.703216,0.783222,723.162917
3,target_next_session_length_sec,train_quantile_04,2082.586408,544.360599,213.034195,444.415236,1345.368588,-0.022715,224.147787,232.808409,136.179630,376.744601,2135.217756,0.800243,750.525545,551.024963,204.708756,434.604197,1332.764433,-0.019498,219.824247,228.776888,131.261193,379.461026,2234.954949,0.804889,768.513202,543.377297,210.539928,440.629530,1335.000450,-0.020051,223.177730,231.806123,134.585042,379.568033,2153.938254,0.800999,750.866008


In [15]:
mode_name = "train_engagement_quantile_035"
cfg = next(cfg for cfg in REG_MODES if cfg["mode"] == mode_name)

for target_col, p in packs.items():
    model, ptr, pva, pte, fit_sec = fit_regression_mode(p, cfg)
    reg_models[(target_col, cfg["mode"])] = model

    row = {
        "target": target_col,
        "mode": cfg["mode"],
        "fit_sec": fit_sec,
    }

    for prefix, y, pred in [
        ("train_", p.y_train, ptr),
        ("val_", p.y_val, pva),
        ("test_", p.y_test, pte),
    ]:
        for k, v in metric_pack(y, pred).items():
            row[prefix + k] = v

    reg_rows.append(row)

reg_res = (
    pd.DataFrame(reg_rows)
    .drop_duplicates(subset=["target", "mode"], keep="last")
    .sort_values(["target", "test_mae"])
    .reset_index(drop=True)
)

reg_res

,target,mode,fit_sec,train_mae,train_medae,train_p70_abs_error,train_p90_abs_error,train_r2,train_product_mae,train_engagement_risk_mae,train_small_mae,train_normal_mae,train_long_mae,train_wmape,train_tail_weighted_mae_q90_w2,val_mae,val_medae,val_p70_abs_error,val_p90_abs_error,val_r2,val_product_mae,val_engagement_risk_mae,val_small_mae,val_normal_mae,val_long_mae,val_wmape,val_tail_weighted_mae_q90_w2,test_mae,test_medae,test_p70_abs_error,test_p90_abs_error,test_r2,test_product_mae,test_engagement_risk_mae,test_small_mae,test_normal_mae,test_long_mae,test_wmape,test_tail_weighted_mae_q90_w2
0,future_sessions_mean_playtime_7d,train_capped_target,807.748157,248.659596,144.811009,264.541877,563.002237,0.355720,169.939350,171.951615,142.974194,185.454975,935.795151,0.422667,312.081477,250.502984,141.573665,261.335829,575.978064,0.358644,167.319364,169.490673,130.728380,193.018711,937.977335,0.428136,315.144132,251.024264,142.663518,261.291366,575.839577,0.367384,166.815525,169.964985,133.520267,190.776847,941.949903,0.426110,316.178570
1,future_sessions_mean_playtime_7d,train_quantile_04,755.880693,257.219048,140.175254,263.728715,602.212944,0.285027,157.819646,165.452109,111.268523,197.864910,1047.087701,0.437216,330.034060,259.819766,137.454177,262.899276,617.177723,0.286597,156.306946,163.386136,100.811035,206.294378,1050.437437,0.444059,333.978900,260.680396,138.524799,264.295701,617.949029,0.295100,156.060127,164.059470,102.564301,204.892126,1054.275137,0.442501,335.326140
2,future_sessions_mean_playtime_7d,train_engagement_quantile_035,697.997511,265.472009,140.893341,271.672837,631.728333,0.239525,157.450012,166.301237,97.348320,209.990718,1101.933803,0.451245,342.544457,267.287743,139.228632,271.695181,642.527381,0.249854,157.630004,165.504178,90.253179,219.138843,1086.240362,0.456823,344.028620,267.290940,139.857268,272.042556,641.255267,0.264964,157.874266,166.757399,92.392123,218.457186,1075.788720,0.453722,343.269983
3,target_next_session_length_sec,train_capped_target,1855.794127,532.522333,247.207745,453.982670,1213.471391,0.026725,252.973451,262.715633,212.946364,310.771503,1965.238078,0.782840,723.136644,538.646053,235.876289,444.077759,1205.657169,0.022543,246.627598,256.597849,201.900684,316.468433,2064.185521,0.786807,740.522573,531.317647,242.942723,450.142621,1206.747477,0.027709,250.872921,260.609509,208.225719,315.559087,1983.703216,0.783222,723.162917
4,target_next_session_length_sec,train_quantile_04,2082.586408,544.360599,213.034195,444.415236,1345.368588,-0.022715,224.147787,232.808409,136.179630,376.744601,2135.217756,0.800243,750.525545,551.024963,204.708756,434.604197,1332.764433,-0.019498,219.824247,228.776888,131.261193,379.461026,2234.954949,0.804889,768.513202,543.377297,210.539928,440.629530,1335.000450,-0.020051,223.177730,231.806123,134.585042,379.568033,2153.938254,0.800999,750.866008
5,target_next_session_length_sec,train_engagement_quantile_035,2130.725805,557.212340,203.518601,463.535648,1408.648037,-0.053758,220.620595,228.301791,109.537806,418.092504,2209.526818,0.819136,769.444995,563.695386,196.920750,451.460412,1393.632009,-0.041287,216.927576,224.907499,107.231494,418.930482,2308.506292,0.823396,787.187176,555.915860,201.703525,458.911098,1396.267161,-0.048969,220.101924,227.632257,109.492620,419.259352,2226.901447,0.819483,769.384776


## Витрины лучших регрессионных моделей

Единственного победителя нет. Показываем лидеров по обычному MAE, ProductMAE и EngagementRiskMAE.

В итоговых таблицах рядом с MAE обязательно остается R2.

In [16]:
show_cols = [
    'target', 'mode', 'fit_sec',
    'train_mae', 'val_mae', 'test_mae', 'test_r2',
    'test_medae', 'test_p70_abs_error', 'test_p90_abs_error',
    'test_product_mae', 'test_engagement_risk_mae',
    'test_small_mae', 'test_normal_mae', 'test_long_mae',
    'test_wmape', 'test_tail_weighted_mae_q90_w2',
]
reg_res[show_cols]

,target,mode,fit_sec,train_mae,val_mae,test_mae,test_r2,test_medae,test_p70_abs_error,test_p90_abs_error,test_product_mae,test_engagement_risk_mae,test_small_mae,test_normal_mae,test_long_mae,test_wmape,test_tail_weighted_mae_q90_w2
0,future_sessions_mean_playtime_7d,train_capped_target,807.748157,248.659596,250.502984,251.024264,0.367384,142.663518,261.291366,575.839577,166.815525,169.964985,133.520267,190.776847,941.949903,0.426110,316.178570
1,future_sessions_mean_playtime_7d,train_quantile_04,755.880693,257.219048,259.819766,260.680396,0.295100,138.524799,264.295701,617.949029,156.060127,164.059470,102.564301,204.892126,1054.275137,0.442501,335.326140
2,future_sessions_mean_playtime_7d,train_engagement_quantile_035,697.997511,265.472009,267.287743,267.290940,0.264964,139.857268,272.042556,641.255267,157.874266,166.757399,92.392123,218.457186,1075.788720,0.453722,343.269983
3,target_next_session_length_sec,train_capped_target,1855.794127,532.522333,538.646053,531.317647,0.027709,242.942723,450.142621,1206.747477,250.872921,260.609509,208.225719,315.559087,1983.703216,0.783222,723.162917
4,target_next_session_length_sec,train_quantile_04,2082.586408,544.360599,551.024963,543.377297,-0.020051,210.539928,440.629530,1335.000450,223.177730,231.806123,134.585042,379.568033,2153.938254,0.800999,750.866008
5,target_next_session_length_sec,train_engagement_quantile_035,2130.725805,557.212340,563.695386,555.915860,-0.048969,201.703525,458.911098,1396.267161,220.101924,227.632257,109.492620,419.259352,2226.901447,0.819483,769.384776


In [17]:
best_mae = reg_res.sort_values(['test_mae', 'test_r2'], ascending=[True, False]).groupby('target', as_index=False).first()
best_product = reg_res.sort_values('test_product_mae').groupby('target', as_index=False).first()
best_engagement = reg_res.sort_values('test_engagement_risk_mae').groupby('target', as_index=False).first()

best_mae[['target', 'mode', 'test_mae', 'test_r2', 'test_product_mae', 'test_engagement_risk_mae']]

,target,mode,test_mae,test_r2,test_product_mae,test_engagement_risk_mae
0,future_sessions_mean_playtime_7d,train_capped_target,251.024264,0.367384,166.815525,169.964985
1,target_next_session_length_sec,train_capped_target,531.317647,0.027709,250.872921,260.609509


In [18]:
best_product[['target', 'mode', 'test_mae', 'test_r2', 'test_product_mae', 'test_engagement_risk_mae']]

,target,mode,test_mae,test_r2,test_product_mae,test_engagement_risk_mae
0,future_sessions_mean_playtime_7d,train_quantile_04,260.680396,0.295100,156.060127,164.059470
1,target_next_session_length_sec,train_engagement_quantile_035,555.915860,-0.048969,220.101924,227.632257


## Отдельная модель churn 7d

Для риска ухода нужна классификация, а не регрессия. Гиперпараметры и class weights берем из финального churn блока предыдущего ноутбука.

Модель оценивается по ROC AUC, PR AUC, logloss, Brier score, precision, recall и F1.

In [1]:
# Bootstrap для запуска churn-блока без пересчета верхних ячеек
# Если ядро перезапустилось, можно начать выполнение с этой ячейки и дальше.
# Верхние тяжелые CatBoost-регрессии повторно не запускаются.

from pathlib import Path
import json
import sys
import time

import numpy as np
import pandas as pd

repo_root = Path.cwd()
if repo_root.name != 'ml_in_gamedev_project':
    for p in [repo_root, *repo_root.parents]:
        if p.name == 'ml_in_gamedev_project':
            repo_root = p
            break

if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

from catboost import CatBoostClassifier, CatBoostRegressor
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score,
)

from preprocessing.preprocessing import (
    CRM_TARGET,
    DEFAULT_TARGET,
    build_feature_list,
    load_data,
    prepare_for_targets,
)
from crm.crm_logic import build_crm_payload, crm_risk_score

pd.set_option('display.max_columns', 300)

if 'MAX_ROWS' not in globals():
    MAX_ROWS = None
if 'SAVE_MODELS' not in globals():
    SAVE_MODELS = True
if 'MODEL_VERSION' not in globals():
    MODEL_VERSION = 'crm_full_v1'

if 'FINAL_REG_PARAMS' not in globals():
    FINAL_REG_PARAMS = {
        'depth': 7,
        'learning_rate': 0.05,
        'l2_leaf_reg': 3,
        'iterations': 1100,
        'bagging_temperature': 1.0,
        'subsample': 0.70,
        'random_strength': 2.0,
        'border_count': 64,
        'od_type': 'Iter',
        'od_wait': 80,
        'random_seed': 42,
        'verbose': False,
        'thread_count': -1,
    }

if 'add_crm_retention_targets' not in globals():
    def add_crm_retention_targets(
        df_in,
        installation_col='installation_id',
        time_col='start',
        duration_col='duration_seconds',
        horizon_days=7,
    ):
        x = df_in.copy()
        x[time_col] = pd.to_datetime(x[time_col], errors='coerce')
        x = x[x[time_col].notna() & x[installation_col].notna()].copy()
        x = x.sort_values([installation_col, time_col]).reset_index(drop=True)

        delta_ns = np.int64(horizon_days * 24 * 3600 * 1_000_000_000)
        dur = pd.to_numeric(x[duration_col], errors='coerce').fillna(0.0).to_numpy(dtype=np.float64)
        fut_play = np.zeros(len(x), dtype=np.float64)
        fut_cnt = np.zeros(len(x), dtype=np.int32)

        for _, idx in x.groupby(installation_col, sort=False).indices.items():
            idx = np.asarray(idx, dtype=np.int64)
            t = x.iloc[idx][time_col].values.astype('datetime64[ns]').astype(np.int64)
            d = dur[idx]
            n = len(idx)

            nxt = np.arange(n, dtype=np.int64) + 1
            right = np.searchsorted(t, t + delta_ns, side='right')
            fut_cnt[idx] = np.maximum(right - nxt, 0)

            csum = np.concatenate(([0.0], np.cumsum(d)))
            fut_play[idx] = csum[right] - csum[np.minimum(nxt, n)]

        max_t = x[time_col].max().value
        observed = x[time_col].values.astype('datetime64[ns]').astype(np.int64) <= max_t - delta_ns
        mean7 = np.divide(fut_play, np.maximum(fut_cnt, 1))

        x['future_sessions_count_7d'] = np.where(observed, fut_cnt, np.nan)
        x[CRM_TARGET] = np.where(observed, mean7, np.nan)
        x['churn_7d'] = np.where(observed, (fut_cnt == 0).astype(int), np.nan)
        return x

if 'crm_df' not in globals() or 'packs' not in globals():
    df = load_data()
    crm_df = add_crm_retention_targets(df)
    limit = 0 if MAX_ROWS is None else int(MAX_ROWS)
    packs = prepare_for_targets(
        crm_df,
        target_cols=[DEFAULT_TARGET, CRM_TARGET],
        max_rows=limit,
    )

if 'reg_models' not in globals():
    reg_models = {}
if 'reg_res' not in globals():
    reg_res = pd.DataFrame()

pd.DataFrame([
    {
        'target': t,
        'train_rows': len(pack.x_train),
        'val_rows': len(pack.x_val),
        'test_rows': len(pack.x_test),
        'features': len(pack.feature_cols),
        'cat_features': len(pack.cat_cols),
    }
    for t, pack in packs.items()
])

,target,train_rows,val_rows,test_rows,features,cat_features
0,target_next_session_length_sec,2406968,515779,515780,74,14
1,future_sessions_mean_playtime_7d,2357551,505189,505190,74,14


In [2]:
def prepare_churn_data(df_in, max_rows=None):
    x = df_in[df_in['start'].notna() & df_in['churn_7d'].notna()].copy()
    x['start'] = pd.to_datetime(x['start'], errors='coerce')
    x = x.sort_values('start').reset_index(drop=True)
    if max_rows is not None and len(x) > max_rows:
        x = x.tail(int(max_rows)).reset_index(drop=True)

    n = len(x)
    i1, i2 = int(n * 0.70), int(n * 0.85)
    tr, va, te = x.iloc[:i1].copy(), x.iloc[i1:i2].copy(), x.iloc[i2:].copy()

    feature_cols = build_feature_list(x, target_col='churn_7d')
    feature_cols = [c for c in feature_cols if c != 'churn_7d']
    xtr, xva, xte = tr[feature_cols].copy(), va[feature_cols].copy(), te[feature_cols].copy()
    ytr = tr['churn_7d'].astype(int).values
    yva = va['churn_7d'].astype(int).values
    yte = te['churn_7d'].astype(int).values

    num_cols = xtr.select_dtypes(include=[np.number, 'bool']).columns.tolist()
    cat_cols = [c for c in feature_cols if c not in num_cols]
    medians = xtr[num_cols].median() if num_cols else pd.Series(dtype=float)

    if num_cols:
        xtr[num_cols] = xtr[num_cols].fillna(medians)
        xva[num_cols] = xva[num_cols].fillna(medians)
        xte[num_cols] = xte[num_cols].fillna(medians)
    if cat_cols:
        xtr[cat_cols] = xtr[cat_cols].astype('object').fillna('unknown')
        xva[cat_cols] = xva[cat_cols].astype('object').fillna('unknown')
        xte[cat_cols] = xte[cat_cols].astype('object').fillna('unknown')

    prep = {'feature_cols': feature_cols, 'num_cols': num_cols, 'cat_cols': cat_cols, 'medians': medians}
    return xtr, xva, xte, ytr, yva, yte, prep


def classification_metrics(y_true, prob, threshold=0.5):
    y = np.asarray(y_true, dtype=int)
    p = np.clip(np.asarray(prob, dtype=float), 1e-6, 1 - 1e-6)
    pred = (p >= threshold).astype(int)
    return {
        'roc_auc': float(roc_auc_score(y, p)),
        'pr_auc': float(average_precision_score(y, p)),
        'logloss': float(log_loss(y, p, labels=[0, 1])),
        'brier': float(brier_score_loss(y, p)),
        'accuracy': float(accuracy_score(y, pred)),
        'balanced_accuracy': float(balanced_accuracy_score(y, pred)),
        'precision': float(precision_score(y, pred, zero_division=0)),
        'recall': float(recall_score(y, pred, zero_division=0)),
        'f1': float(f1_score(y, pred, zero_division=0)),
        'true_pos_rate': float(y.mean()),
        'pred_pos_rate': float(pred.mean()),
    }

In [3]:
xtr_ch, xva_ch, xte_ch, ytr_ch, yva_ch, yte_ch, churn_prep = prepare_churn_data(crm_df, MAX_ROWS)

CHURN_PARAMS = {
    'depth': 6,
    'learning_rate': 0.05,
    'l2_leaf_reg': 5,
    'iterations': 500,
    'bagging_temperature': 0.5,
    'subsample': 0.85,
    'random_strength': 1.0,
    'border_count': 128,
    'od_type': 'Iter',
    'od_wait': 80,
    'loss_function': 'Logloss',
    'eval_metric': 'AUC',
    'class_weights': [1.0, 1.5],
    'random_seed': 42,
    'verbose': False,
    'thread_count': -1,
}

churn_model = CatBoostClassifier(**CHURN_PARAMS)
t0 = time.time()
churn_model.fit(
    xtr_ch,
    ytr_ch,
    cat_features=churn_prep['cat_cols'],
    eval_set=(xva_ch, yva_ch),
    use_best_model=True,
)
churn_fit_sec = time.time() - t0

churn_val_prob = churn_model.predict_proba(xva_ch)[:, 1]
churn_test_prob = churn_model.predict_proba(xte_ch)[:, 1]

churn_metrics = pd.DataFrame([
    {'split': 'val', 'fit_sec': churn_fit_sec, **classification_metrics(yva_ch, churn_val_prob)},
    {'split': 'test', 'fit_sec': churn_fit_sec, **classification_metrics(yte_ch, churn_test_prob)},
])
churn_metrics

,split,fit_sec,roc_auc,pr_auc,logloss,brier,accuracy,balanced_accuracy,precision,recall,f1,true_pos_rate,pred_pos_rate
0,val,415.463473,0.836412,0.371018,0.226743,0.064193,0.917100,0.631757,0.476380,0.291600,0.361760,0.080570,0.049318
1,test,415.463473,0.830531,0.376181,0.244158,0.069846,0.908852,0.638730,0.471694,0.311016,0.374863,0.087868,0.057937


## Интервальные модели для CRM

Для прикладного ответа одного точечного прогноза недостаточно. MultiQuantile одновременно оценивает p30, p50 и p70.

- p30 используется как cautious prediction
- p50 используется как основной прогноз
- p30 и p70 задают рабочий интервал неопределенности

In [6]:
INTERVAL_PARAMS = {
    **FINAL_REG_PARAMS,
    'iterations': 200,
    'loss_function': 'MultiQuantile:alpha=0.3,0.5,0.7',
    'eval_metric': 'MultiQuantile:alpha=0.3,0.5,0.7',
}

interval_models = {}

for target_col, p in packs.items():
    print('start interval model:', target_col)
    print('train rows:', len(p.x_train), 'val rows:', len(p.x_val), 'cat cols:', len(p.cat_cols))

    t0 = time.time()
    m = CatBoostRegressor(**INTERVAL_PARAMS)
    m.fit(
        p.x_train,
        p.y_train,
        cat_features=p.cat_cols,
        eval_set=(p.x_val, p.y_val),
        use_best_model=True,
    )

    interval_models[target_col] = m
    print('done:', target_col, 'fit_sec:', round(time.time() - t0, 1))

start interval model: target_next_session_length_sec
train rows: 2406968 val rows: 515779 cat cols: 14
done: target_next_session_length_sec fit_sec: 674.6
start interval model: future_sessions_mean_playtime_7d
train rows: 2357551 val rows: 505189 cat cols: 14
done: future_sessions_mean_playtime_7d fit_sec: 651.3


In [7]:
INTERVAL_PARAMS = {
    **FINAL_REG_PARAMS,
    'iterations': 800,
    'loss_function': 'MultiQuantile:alpha=0.3,0.5,0.7',
    'eval_metric': 'MultiQuantile:alpha=0.3,0.5,0.7',
}

interval_models = {}

for target_col, p in packs.items():
    print('start interval model:', target_col)
    print('train rows:', len(p.x_train), 'val rows:', len(p.x_val), 'cat cols:', len(p.cat_cols))

    t0 = time.time()
    m = CatBoostRegressor(**INTERVAL_PARAMS)
    m.fit(
        p.x_train,
        p.y_train,
        cat_features=p.cat_cols,
        eval_set=(p.x_val, p.y_val),
        use_best_model=True,
    )

    interval_models[target_col] = m
    print('done:', target_col, 'fit_sec:', round(time.time() - t0, 1))

start interval model: target_next_session_length_sec
train rows: 2406968 val rows: 515779 cat cols: 14
done: target_next_session_length_sec fit_sec: 2672.9
start interval model: future_sessions_mean_playtime_7d
train rows: 2357551 val rows: 505189 cat cols: 14
done: future_sessions_mean_playtime_7d fit_sec: 1350.4


## Сохранение итоговых моделей

После полного запуска сохраняем обученные модели локально в папку models. Эти файлы можно не добавлять в git, если они получатся тяжелыми.

In [8]:
model_dir = repo_root / 'models' / 'participant_1'

if SAVE_MODELS:
    model_dir.mkdir(parents=True, exist_ok=True)

    for (target_col, mode), model in reg_models.items():
        model.save_model(str(model_dir / f'{target_col}__{mode}.cbm'))

    churn_model.save_model(str(model_dir / 'churn_7d__train_quantile_04.cbm'))

    for target_col, model in interval_models.items():
        model.save_model(str(model_dir / f'{target_col}__multiquantile_p30_p50_p70.cbm'))

    metadata = {
        'model_version': MODEL_VERSION,
        'max_rows': MAX_ROWS,
        'regression_params': FINAL_REG_PARAMS,
        'churn_params': CHURN_PARAMS,
        'interval_params': INTERVAL_PARAMS,
        'regression_metrics': reg_res.to_dict('records'),
        'churn_metrics': churn_metrics.to_dict('records'),
    }
    (model_dir / 'metadata.json').write_text(
        json.dumps(metadata, ensure_ascii=False, indent=2, default=str),
        encoding='utf-8',
    )

model_dir

PosixPath('/Users/avals282006/Desktop/project/ml_in_gamedev_project/models/participant_1')

## Подготовка признаков для CRM скоринга

Для демонстрации берем последние доступные записи уникальных игроков. Используются только признаки, известные в момент прогноза.

Фактические значения будущих таргетов в итоговый CRM ответ не попадают.

In [9]:
def prepare_score_features(df_in, prep):
    x = df_in[prep.feature_cols].copy()
    if prep.num_cols:
        x[prep.num_cols] = x[prep.num_cols].fillna(prep.x_train[prep.num_cols].median())
    if prep.cat_cols:
        x[prep.cat_cols] = x[prep.cat_cols].astype('object').fillna('unknown')
    return x


def prepare_churn_score_features(df_in, prep):
    x = df_in[prep['feature_cols']].copy()
    if prep['num_cols']:
        x[prep['num_cols']] = x[prep['num_cols']].fillna(prep['medians'])
    if prep['cat_cols']:
        x[prep['cat_cols']] = x[prep['cat_cols']].astype('object').fillna('unknown')
    return x


latest_players = (
    crm_df.sort_values('start')
    .drop_duplicates('installation_id', keep='last')
    .tail(1000)
    .reset_index(drop=True)
)

x_ns = prepare_score_features(latest_players, packs[DEFAULT_TARGET])
x_wk = prepare_score_features(latest_players, packs[CRM_TARGET])
x_ch = prepare_churn_score_features(latest_players, churn_prep)

## CRM таблица

В финальной витрине объединяем:

- интервальный прогноз следующей сессии
- интервальный недельный прогноз
- вероятность churn 7d
- CRM-risk score

Эта таблица уже подходит как источник для дальнейшей сегментации и формирования сценариев коммуникации.

In [10]:
ns_interval = np.maximum(interval_models[DEFAULT_TARGET].predict(x_ns), 0.0)
wk_interval = np.maximum(interval_models[CRM_TARGET].predict(x_wk), 0.0)
churn_probability = churn_model.predict_proba(x_ch)[:, 1]

crm_scores = pd.DataFrame({
    'player_id': latest_players['installation_id'].astype(str),
    'next_session_p30_sec': ns_interval[:, 0],
    'next_session_p50_sec': ns_interval[:, 1],
    'next_session_p70_sec': ns_interval[:, 2],
    'weekly_mean_p30_sec': wk_interval[:, 0],
    'weekly_mean_p50_sec': wk_interval[:, 1],
    'weekly_mean_p70_sec': wk_interval[:, 2],
    'churn_probability_7d': churn_probability,
})
crm_scores['crm_risk_score'] = [
    crm_risk_score(c, w)
    for c, w in zip(crm_scores['churn_probability_7d'], crm_scores['weekly_mean_p50_sec'])
]
crm_scores.sort_values('crm_risk_score', ascending=False).head(20)

,player_id,next_session_p30_sec,next_session_p50_sec,next_session_p70_sec,weekly_mean_p30_sec,weekly_mean_p50_sec,weekly_mean_p70_sec,churn_probability_7d,crm_risk_score
101,e8c9e6189e8e452da227770a8f85a32a,16.874722,46.991588,174.182195,68.770158,220.870025,439.555620,0.821839,0.819480
197,71687c4cacf74869820f0a124d87f530,52.302896,123.835150,335.752506,63.335032,201.250543,423.361251,0.723071,0.766759
351,87bd5a5130f743ec8eb27d15b0545a13,47.897778,105.900811,298.031149,63.556259,201.603280,423.796785,0.718724,0.764034
781,fea08aeb58224a8e8026527acedecb16,48.328913,115.116225,333.378035,71.625021,217.258658,445.820986,0.723492,0.761676
584,86ed49617c664dfe9f095cd113e065ea,42.286303,114.726660,346.614523,72.756921,218.257996,442.542221,0.717586,0.757799
185,eced30d52fab49d2b8ab5d2e094d49fd,50.987558,124.499179,345.528099,67.025696,213.970458,456.451107,0.714602,0.757438
242,cafec77bf4374bdeaefb5db02e6cd90c,31.954489,85.903450,242.353554,84.471763,216.051953,428.467757,0.714917,0.756933
323,091d484806aa404eab3bfd9ffe17d391,48.297540,125.708486,357.014611,73.794671,221.501925,452.872790,0.717080,0.756414
402,a2a5308586434be2b402195c6584cfbe,59.063393,154.448836,432.006946,78.593594,231.346578,463.353648,0.720520,0.755196
805,d986af7623f3436492a3943de8bde17a,54.876998,135.747735,375.248060,71.269325,219.338688,452.221194,0.713483,0.754977


## Примеры полного CRM payload

Общий модуль crm_logic добавляет сегменты, short-session риск, неопределенность, флаг avoid_early_ad и рекомендуемый сценарий.

In [11]:
payloads = []
for row in crm_scores.sort_values('crm_risk_score', ascending=False).head(5).itertuples():
    payloads.append(
        build_crm_payload(
            player_id=row.player_id,
            next_session={
                'predicted_length_sec': row.next_session_p50_sec,
                'cautious_length_sec': row.next_session_p30_sec,
                'interval_low_sec': row.next_session_p30_sec,
                'interval_median_sec': row.next_session_p50_sec,
                'interval_high_sec': row.next_session_p70_sec,
            },
            weekly_engagement={
                'predicted_mean_session_length_sec': row.weekly_mean_p50_sec,
                'cautious_mean_session_length_sec': row.weekly_mean_p30_sec,
                'interval_low_sec': row.weekly_mean_p30_sec,
                'interval_median_sec': row.weekly_mean_p50_sec,
                'interval_high_sec': row.weekly_mean_p70_sec,
                'churn_probability_7d': row.churn_probability_7d,
            },
            model_version=MODEL_VERSION,
        )
    )

print(json.dumps(payloads, ensure_ascii=False, indent=2, default=float))

[
  {
    "player_id": "e8c9e6189e8e452da227770a8f85a32a",
    "next_session": {
      "predicted_length_sec": 46.99158755857209,
      "cautious_length_sec": 16.874722078578145,
      "interval_low_sec": 16.874722078578145,
      "interval_median_sec": 46.99158755857209,
      "interval_high_sec": 174.18219469521694
    },
    "weekly_engagement": {
      "predicted_mean_session_length_sec": 220.87002496476393,
      "cautious_mean_session_length_sec": 68.77015799872032,
      "interval_low_sec": 68.77015799872032,
      "interval_median_sec": 220.87002496476393,
      "interval_high_sec": 439.55561963427135,
      "churn_probability_7d": 0.8218387172022614
    },
    "segments": {
      "next_session_segment": "small",
      "weekly_engagement_segment": "small",
      "crm_segment": "at_risk_short"
    },
    "risk_flags": {
      "short_session_risk": "high",
      "prediction_uncertainty": "high",
      "avoid_early_ad": true
    },
    "recommended_scenario": "retention_offer",
  

## Финальный вывод

После полного выполнения этот ноутбук становится итоговой точкой части участника 1.

В нем сохраняются:

- метрики трех регрессионных стратегий на двух таргетах
- train, validation и test метрики для контроля переобучения и временного сдвига
- качество churn классификатора
- интервальные прогнозы p30, p50 и p70
- готовая CRM таблица и несколько полных payload

Правила выбора остаются прежними:

- train_capped_target для точного прогноза секунд
- train_quantile_04 как компромиссный CRM режим
- train_engagement_quantile_035 для осторожных retention решений